# MedGuard AI — Module 1: Prescription Analysis (v2)
### Donut OCR + BioBERT NER Pipeline with Gradio Interface


## Install Dependencies

In [ ]:
!pip install torch torchvision transformers datasets gradio \
             Pillow numpy sentencepiece seqeval jiwer \
             huggingface-hub pytorch-lightning albumentations -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 75.3 MB/s eta 0:00:00


## Imports

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import gradio as gr

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


---
# Donut OCR: Download & Load

## Download Donut Prescription Model

In [ ]:
DONUT_MODEL_DIR = "model"
DONUT_REPO_ID   = "chinmays18/medical-prescription-ocr"

if not os.path.exists(DONUT_MODEL_DIR):
    print(f"Downloading Donut model (~800MB) from {DONUT_REPO_ID}...")
    os.makedirs(DONUT_MODEL_DIR, exist_ok=True)
    snapshot_download(
        repo_id=DONUT_REPO_ID,
        local_dir=DONUT_MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print("Donut model downloaded successfully.")
else:
    print("Donut model already downloaded. Skipping.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Donut model downloaded successfully.


## Load Donut Model

In [ ]:
donut_processor = DonutProcessor.from_pretrained(DONUT_MODEL_DIR)
donut_model     = VisionEncoderDecoderModel.from_pretrained(DONUT_MODEL_DIR)
donut_model     = donut_model.to(DEVICE)
donut_model.eval()
print("Donut OCR model loaded successfully.")

The image processor of type `DonutImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Donut OCR model loaded successfully.


## Test Donut OCR on a Sample Image

In [ ]:
def extract_text_donut(image: Image.Image) -> str:
    """Extract text from a prescription image using Donut OCR."""
    image    = image.convert("RGB")
    encoding = donut_processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        generated_ids = donut_model.generate(
            encoding.pixel_values,
            max_length=512,
            num_beams=1,
            early_stopping=True,
            decoder_start_token_id=donut_processor.tokenizer.convert_tokens_to_ids("<s_ocr>")
        )

    text = donut_processor.tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0].strip()
    return text


# Quick test with a blank white image just to confirm the model runs
test_image  = Image.new("RGB", (800, 600), color="white")
test_output = extract_text_donut(test_image)
print("Donut OCR test output:", test_output)
print("Donut OCR is working correctly.")

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Donut OCR test output: 
Donut OCR is working correctly.


---
# BioBERT NER: Load, Fine-tune on BC5CDR, Evaluate

## Load BC5CDR Dataset

In [ ]:
# Load BC5CDR directly from Parquet to bypass legacy script issue
bc5cdr = load_dataset("parquet", data_files={
    "train":      "hf://datasets/tner/bc5cdr@~parquet/bc5cdr/train/0000.parquet",
    "validation": "hf://datasets/tner/bc5cdr@~parquet/bc5cdr/validation/0000.parquet"
})

print(bc5cdr)
print("\nSample entry:")
print(bc5cdr["train"][0])

0000.parquet:   0%|          | 0.00/367k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/364k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 5330
    })
})

Sample entry:
{'tokens': ['Naloxone', 'reverses', 'the', 'antihypertensive', 'effect', 'of', 'clonidine', '.'], 'tags': [1, 0, 0, 0, 0, 0, 1, 0]}


## Load Pretrained BioBERT

In [ ]:
# BC5CDR BIO labels — hardcoded since Parquet has no ClassLabel metadata
LABEL_LIST = ["O", "B-Chemical", "I-Chemical", "B-Disease", "I-Disease"]
NUM_LABELS  = len(LABEL_LIST)
label2id    = {l: i for i, l in enumerate(LABEL_LIST)}
id2label    = {i: l for i, l in enumerate(LABEL_LIST)}

ner_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
ner_model     = AutoModelForTokenClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2",
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
ner_model = ner_model.to(DEVICE)
print(f"BioBERT loaded successfully.")
print(f"Labels: {LABEL_LIST}")

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored 

BioBERT loaded successfully.
Labels: ['O', 'B-Chemical', 'I-Chemical', 'B-Disease', 'I-Disease']


## Tokenize BC5CDR for NER

In [ ]:
def tokenize_and_align(examples):
    tokenized = ner_tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding="max_length"
    )
    all_labels = []
    for i, labels in enumerate(examples["tags"]):
        word_ids     = tokenized.word_ids(batch_index=i)
        label_ids    = []
        prev_word_id = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)          # Special tokens — ignored in loss
            elif wid != prev_word_id:
                label_ids.append(labels[wid])   # First subword → real label
            else:
                label_ids.append(-100)          # Subsequent subwords → ignored
            prev_word_id = wid
        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


tokenized_bc5cdr = bc5cdr.map(tokenize_and_align, batched=True)
tokenized_bc5cdr = tokenized_bc5cdr.remove_columns(["tokens", "tags"])
tokenized_bc5cdr.set_format("torch")
print("Tokenization complete.")
print(tokenized_bc5cdr)

Map:   0%|          | 0/5228 [00:00<?, ? examples/s]

Map:   0%|          | 0/5330 [00:00<?, ? examples/s]

Tokenization complete.
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5330
    })
})


## Fine-tune BioBERT on BC5CDR

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                p_seq.append(id2label[p])
                l_seq.append(id2label[l])
        true_preds.append(p_seq)
        true_labels.append(l_seq)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall"   : recall_score(true_labels, true_preds),
        "f1"       : f1_score(true_labels, true_preds),
    }


training_args = TrainingArguments(
    output_dir                  = "./ner_model",
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-5,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    logging_steps               = 50,
    report_to                   = "none"
)

data_collator = DataCollatorForTokenClassification(ner_tokenizer)

trainer = Trainer(
    model            = ner_model,
    args             = training_args,
    train_dataset    = tokenized_bc5cdr["train"],
    eval_dataset     = tokenized_bc5cdr["validation"],
    processing_class = ner_tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics
)

trainer.train()
print("\nBioBERT NER fine-tuning complete.")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.077243,0.072854,0.869628,0.882625,0.876078
2,0.042017,0.072784,0.882739,0.884818,0.883778
3,0.021848,0.082319,0.875875,0.894728,0.885201


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


BioBERT NER fine-tuning complete.


## Evaluate BioBERT NER

In [ ]:
results = trainer.evaluate()
print("\nNER Evaluation Results:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


NER Evaluation Results:
  eval_loss: 0.0728
  eval_precision: 0.8827
  eval_recall: 0.8847
  eval_f1: 0.8837
  eval_runtime: 46.2830
  eval_samples_per_second: 115.1610
  eval_steps_per_second: 7.2160
  epoch: 3.0000


---
# Full Pipeline: Donut OCR → BioBERT NER

## Define Full Inference Pipeline

In [ ]:
# Medical keywords for prescription validation
MEDICAL_KEYWORDS = [
    "prescribed", "take", "mg", "ml", "capsule", "tablet", "dosage",
    "dr", "doctor", "patient", "medication", "apply", "clinic",
    "pharmacy", "rx", "dose", "medicine", "drug", "daily", "twice",
    "morning", "night", "oral", "injection", "syrup", "drops"
]


def is_prescription(text: str) -> tuple:
    """Simple heuristic check — is this text from a prescription?"""
    text_lower = text.lower()
    matches    = [kw for kw in MEDICAL_KEYWORDS if kw in text_lower]
    confidence = min(len(matches) / 5, 1.0)  # Normalise to 0-1
    verdict    = "Medical Prescription" if len(matches) >= 2 else "Not a Prescription"
    return verdict, round(confidence, 2), matches


def extract_entities(text: str) -> dict:
    """Run fine-tuned BioBERT NER on extracted text."""
    ner_model.eval()
    inputs = ner_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(DEVICE)

    with torch.no_grad():
        outputs = ner_model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)[0]
    tokens      = ner_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    entities      = {"Drugs / Chemicals": [], "Diseases": []}
    current_words = []
    current_label = None

    for token, pred_id in zip(tokens, predictions):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        label = id2label[pred_id.item()]

        if label.startswith("B-"):
            # Save previous entity if exists
            if current_words and current_label:
                entity_text = ner_tokenizer.convert_tokens_to_string(current_words).strip()
                if "Chemical" in current_label:
                    entities["Drugs / Chemicals"].append(entity_text)
                elif "Disease" in current_label:
                    entities["Diseases"].append(entity_text)
            current_words = [token]
            current_label = label

        elif label.startswith("I-") and current_words:
            current_words.append(token)

        else:
            # Save entity on O label
            if current_words and current_label:
                entity_text = ner_tokenizer.convert_tokens_to_string(current_words).strip()
                if "Chemical" in current_label:
                    entities["Drugs / Chemicals"].append(entity_text)
                elif "Disease" in current_label:
                    entities["Diseases"].append(entity_text)
            current_words = []
            current_label = None

    # Remove duplicates and clean
    for key in entities:
        entities[key] = list(set([e for e in entities[key] if len(e) > 1]))

    return entities


def run_full_pipeline(image: Image.Image) -> tuple:
    """Full pipeline: Image → Donut OCR → BioBERT NER → Structured Output."""

    # Stage 1 — Donut OCR
    extracted_text = extract_text_donut(image)

    # Stage 2 — Prescription validation
    verdict, confidence, matched_keywords = is_prescription(extracted_text)

    # Stage 3 — BioBERT NER
    entities = extract_entities(extracted_text)

    # Format structured output
    structured = f"""PRESCRIPTION ANALYSIS REPORT
{'='*45}

CLASSIFICATION : {verdict}
CONFIDENCE     : {confidence}
KEYWORDS FOUND : {', '.join(matched_keywords) if matched_keywords else 'None'}

EXTRACTED ENTITIES:
  Drugs / Chemicals : {', '.join(entities['Drugs / Chemicals']) or 'None detected'}
  Diseases          : {', '.join(entities['Diseases'])          or 'None detected'}

{'='*45}
MedGuard AI | Module 1 | Rushdi 2237950
"""
    return extracted_text, structured


print("Pipeline functions defined successfully.")

Pipeline functions defined successfully.


---
# Gradio Interface

## Launch Gradio UI

In [ ]:
def gradio_predict(image):
    if image is None:
        return "No image uploaded.", "Please upload a prescription image."
    pil_image = Image.fromarray(image) if not isinstance(image, Image.Image) else image
    extracted_text, structured_output = run_full_pipeline(pil_image)
    return extracted_text, structured_output


with gr.Blocks(title="MedGuard AI — Prescription Analysis") as demo:

    gr.Markdown("""
    # MedGuard AI — Prescription Analysis
    **Module 1** | Donut OCR + BioBERT NER Pipeline
    Upload a prescription image to extract and analyse its contents.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                label="Upload Prescription Image",
                type="pil"
            )
            submit_btn = gr.Button(
                "Analyse Prescription",
                variant="primary"
            )

        with gr.Column(scale=1):
            ocr_output = gr.Textbox(
                label="Extracted Text (Donut OCR)",
                lines=6,
                placeholder="Extracted prescription text will appear here..."
            )
            ner_output = gr.Textbox(
                label="Analysis Report (BioBERT NER)",
                lines=14,
                placeholder="Structured analysis report will appear here..."
            )

    submit_btn.click(
        fn      = gradio_predict,
        inputs  = [image_input],
        outputs = [ocr_output, ner_output]
    )

    gr.Markdown("""
    ---
    *MedGuard AI | Module 1 — Prescription Analysis | Rushdi 2237950 / 20220160*
    """)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14b1f0ebc51152c07d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
